# 02 - Bronze to Silver with PySpark SQL

Parse types, normalize metrics, deduplicate events, derive date columns, and quarantine invalid records.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.spark_pipeline import create_spark_session, read_parquet_object, transform_silver
from insightsstream.storage import create_store

spark = create_spark_session("InsightsStream-Notebook-Silver")
store = create_store()

In [ ]:
run = transform_silver(spark=spark, store=store)
print(asdict(run))

silver = read_parquet_object(spark, store, run.silver_key)
silver.createOrReplaceTempView("silver_events")
silver.show(20, truncate=False)
silver.printSchema()

In [ ]:
spark.sql("""
    SELECT
        event_date,
        metric,
        COUNT(*) AS clean_events,
        COUNT(DISTINCT entity_id) AS unique_entities,
        ROUND(SUM(value), 2) AS total_value
    FROM silver_events
    GROUP BY event_date, metric
    ORDER BY event_date, metric
""").show(truncate=False)

print(f"accepted={run.silver_rows}, quarantined={run.rejected_rows}")